# Real vs Generated Text Classifier

Can a small classifier distinguish real FineWeb text from text generated by our Parameter Golf model?
This tests how statistically distinguishable the model's outputs are from real data.

**Approach:**
1. Generate a dataset of real vs generated sentence completions
2. Extract features (token-level stats from the LM itself, plus simple text features)
3. Train a small classifier and analyze what features separate real from generated

In [ ]:
# === CONFIGURATION ===
RUN_ID = "baseline_20k_iters_seq2_20260324_205335"
USE_QUANTIZED = True
SEQ_LEN = 1024
DATA_PATH = "./data/datasets/fineweb10B_sp1024"
TOKENIZER_PATH = "./data/tokenizers/fineweb_1024_bpe.model"

# Dataset parameters
N_SAMPLES = 500  # number of real/generated pairs
PROMPT_LEN = 20  # tokens of real text used as prompt
COMPLETION_LEN = 40  # tokens to generate as completion
SAMPLE_TEMP = 0.8
SAMPLE_TOP_K = 50

In [ ]:
import glob
import importlib.util
import io
import math
import os
import re
import sys
import tempfile
import zlib
from collections import Counter
from pathlib import Path

PLOT_DIR = Path("plots/explore_real_vs_generated")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sentencepiece as spm
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, roc_curve

plt.rcParams["figure.dpi"] = 120
ln2 = math.log(2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# --- Load training script from log and import as module ---
run_dir = Path("models") / RUN_ID
log_txt = run_dir / "log.txt"
assert log_txt.exists(), f"Log file not found: {log_txt}"

if USE_QUANTIZED:
    ckpt_path = run_dir / "model.int8.ptz"
else:
    ckpt_path = run_dir / "model.npz"
assert ckpt_path.exists(), f"Checkpoint not found: {ckpt_path}"

raw_lines = log_txt.read_text().splitlines()

in_main = False
source_end = len(raw_lines)
for i, line in enumerate(raw_lines):
    if line.strip().startswith("if __name__"):
        in_main = True
    if in_main and i > 0:
        stripped = line.strip()
        if stripped and not stripped.startswith("#") and ":" in stripped:
            first_token = stripped.split(":")[0].split("(")[0].split("[")[0]
            if first_token in (
                "step",
                "val_progress",
                "saved_model",
                "serialized_model_int8_zlib",
                "final_int8_zlib_roundtrip",
                "final_int8_zlib_roundtrip_exact",
                "stopping_early",
                "WARNING",
            ):
                source_end = i
                break

main_idx = None
for i, line in enumerate(raw_lines[:source_end]):
    if line.strip().startswith("if __name__"):
        main_idx = i
        break

importable_source = (
    "\n".join(raw_lines[:main_idx]) if main_idx else "\n".join(raw_lines[:source_end])
)

_tmpdir = tempfile.mkdtemp()
_tmp_path = os.path.join(_tmpdir, f"{RUN_ID.replace('-', '_')}_module.py")
with open(_tmp_path, "w") as f:
    f.write(importable_source)

spec = importlib.util.spec_from_file_location("train_module", _tmp_path)
train_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_module)
print(f"Imported training module from {log_txt}")

In [ ]:
# --- Build model and load weights ---
import inspect

hp = train_module.Hyperparameters()
GPT = train_module.GPT
sig = inspect.signature(GPT.__init__)
init_params = [p for p in sig.parameters if p != "self"]

param_map = {
    "vocab_size": hp.vocab_size,
    "num_layers": hp.num_layers,
    "dim": hp.model_dim,
    "num_heads": hp.num_heads,
    "num_kv_heads": hp.num_kv_heads,
    "mlp_mult": hp.mlp_mult,
    "logit_chunk_tokens": getattr(hp, "logit_chunk_tokens", 0),
    "logit_softcap": getattr(hp, "logit_softcap", 30.0),
    "rope_base": getattr(hp, "rope_base", 10000.0),
    "tied_embed_init_std": getattr(hp, "tied_embed_init_std", 0.005),
    "qk_gain_init": getattr(hp, "qk_gain_init", 1.5),
}

kwargs = {}
for p in init_params:
    if p in param_map:
        kwargs[p] = param_map[p]
    elif hasattr(hp, p):
        kwargs[p] = getattr(hp, p)

model = GPT(**kwargs)

if USE_QUANTIZED:
    with open(ckpt_path, "rb") as f:
        quant_blob = f.read()
    quant_obj = torch.load(io.BytesIO(zlib.decompress(quant_blob)), map_location="cpu")
    state_dict = train_module.dequantize_state_dict_int8(quant_obj)
else:
    state_dict = torch.load(ckpt_path, map_location="cpu")

model.load_state_dict(state_dict, strict=True)
model.eval()
num_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {num_params:,} parameters")

In [ ]:
# --- Load tokenizer and validation data ---
sp_tok = spm.SentencePieceProcessor(model_file=TOKENIZER_PATH)
V = sp_tok.vocab_size()
print(f"Tokenizer: vocab_size={V}")

sys.path.insert(0, str(Path(".").resolve()))
from train_gpt import load_validation_tokens, build_sentencepiece_luts

val_pattern = f"{DATA_PATH}/fineweb_val_*.bin"
val_tokens = load_validation_tokens(val_pattern, seq_len=SEQ_LEN)
print(f"Validation tokens: {val_tokens.shape[0]:,}")


# --- Helpers ---
@torch.no_grad()
def get_logits(x):
    """Forward pass: x (1, SEQ_LEN) -> logits (SEQ_LEN, V)"""
    emb = model.tok_emb(x)
    h = F.rms_norm(emb, (emb.size(-1),))
    x0 = h
    skips = []
    for i in range(model.num_encoder_layers):
        h = model.blocks[i](h, x0)
        skips.append(h)
    for i in range(model.num_decoder_layers):
        if skips:
            h = h + model.skip_weights[i].to(dtype=h.dtype)[None, None, :] * skips.pop()
        h = model.blocks[model.num_encoder_layers + i](h, x0)
    h = model.final_norm(h).reshape(-1, h.size(-1))
    if model.tie_embeddings:
        logits = F.linear(h, model.tok_emb.weight)
    else:
        logits = model.lm_head(h)
    return model.logit_softcap * torch.tanh(logits / model.logit_softcap)


@torch.no_grad()
def sample_completion(prompt_ids, max_new_tokens=40, temperature=0.8, top_k=50):
    """Autoregressively sample from prompt token IDs. Returns list of new token IDs."""
    generated = list(prompt_ids)
    for _ in range(max_new_tokens):
        if len(generated) >= SEQ_LEN:
            break
        padded = generated + [0] * (SEQ_LEN - len(generated))
        x = torch.tensor(padded, device=device).unsqueeze(0)
        logits = get_logits(x)
        next_logits = logits[len(generated) - 1] / temperature
        if top_k > 0:
            topk_vals, topk_idx = torch.topk(next_logits, top_k)
            mask = torch.full_like(next_logits, float("-inf"))
            mask.scatter_(0, topk_idx, topk_vals)
            next_logits = mask
        probs = F.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, 1).item()
        if next_token <= 1:  # BOS or pad
            break
        generated.append(next_token)
    return generated[len(prompt_ids) :]


print("Helpers ready.")

In [ ]:
# --- Build dataset: real completions vs generated completions ---
# For each sample:
#   - Pick a random position in val_tokens after a BOS boundary
#   - Use PROMPT_LEN tokens as the prompt
#   - Real completion: the next COMPLETION_LEN tokens from val data
#   - Generated completion: sample COMPLETION_LEN tokens from the model

# Find all BOS positions (document starts)
bos_positions = (val_tokens == 1).nonzero(as_tuple=True)[0].numpy()
# Filter: need at least PROMPT_LEN + COMPLETION_LEN tokens after BOS
valid_bos = bos_positions[
    bos_positions + PROMPT_LEN + COMPLETION_LEN + 1 < val_tokens.shape[0]
]
print(f"Found {len(valid_bos)} valid document starts")

rng = np.random.default_rng(42)
chosen_starts = rng.choice(
    valid_bos, size=min(N_SAMPLES, len(valid_bos)), replace=False
)

dataset = []  # list of dicts with prompt, completion, label, features

for idx, bos_pos in enumerate(tqdm(chosen_starts, desc="Building dataset")):
    bos_pos = int(bos_pos)
    prompt_ids = val_tokens[bos_pos : bos_pos + PROMPT_LEN + 1].tolist()  # includes BOS
    real_ids = val_tokens[
        bos_pos + PROMPT_LEN + 1 : bos_pos + PROMPT_LEN + 1 + COMPLETION_LEN
    ].tolist()

    # Check real completion doesn't cross a document boundary
    if 1 in real_ids:  # BOS in completion = new document
        real_ids = real_ids[: real_ids.index(1)]
    if len(real_ids) < 5:  # too short, skip
        continue

    # Generate completion from same prompt
    gen_ids = sample_completion(
        prompt_ids,
        max_new_tokens=len(real_ids),
        temperature=SAMPLE_TEMP,
        top_k=SAMPLE_TOP_K,
    )
    if len(gen_ids) < 5:
        continue

    # Decode
    prompt_text = sp_tok.decode(prompt_ids)
    real_text = sp_tok.decode(real_ids)
    gen_text = sp_tok.decode(gen_ids)

    dataset.append(
        {
            "prompt_ids": prompt_ids,
            "prompt_text": prompt_text,
            "real_ids": real_ids,
            "real_text": real_text,
            "gen_ids": gen_ids,
            "gen_text": gen_text,
        }
    )

print(f"\nDataset: {len(dataset)} pairs (real + generated)")
print(f"Example prompt: {dataset[0]['prompt_text'][:100]}...")
print(f"  Real: {dataset[0]['real_text'][:100]}")
print(f"  Gen:  {dataset[0]['gen_text'][:100]}")

# --- Save dataset to disk (generation is expensive) ---
import json as _json

save_path = PLOT_DIR / "real_vs_generated_dataset.json"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
serializable = []
for d in dataset:
    serializable.append(
        {
            "prompt_ids": d["prompt_ids"],
            "prompt_text": d["prompt_text"],
            "real_ids": d["real_ids"],
            "real_text": d["real_text"],
            "gen_ids": d["gen_ids"],
            "gen_text": d["gen_text"],
        }
    )
with open(save_path, "w") as f:
    _json.dump(serializable, f)
print(f"Saved {len(dataset)} pairs to {save_path}")


In [ ]:
# --- Extract features for each completion ---
# Features from the model: per-token loss, entropy, rank of target token
# Features from text: repetition, unique token ratio, punctuation density


def extract_features(prompt_ids, completion_ids):
    """Extract a feature dict for a prompt+completion pair."""
    full_ids = prompt_ids + completion_ids
    if len(full_ids) > SEQ_LEN:
        full_ids = full_ids[:SEQ_LEN]
    n = len(full_ids)
    comp_start = len(prompt_ids)  # first completion token position

    # Forward pass
    padded = full_ids + [0] * (SEQ_LEN - n)
    x = torch.tensor(padded, device=device).unsqueeze(0)
    logits = get_logits(x)  # (SEQ_LEN, V)
    log_probs = F.log_softmax(logits.float(), dim=-1)
    probs = log_probs.exp()

    # Per-token stats for the completion region
    losses = []
    entropies = []
    ranks = []
    top1_correct = []

    for pos in range(comp_start, n):
        target_id = full_ids[pos]
        lp = log_probs[pos - 1, target_id].item()
        losses.append(-lp)

        p = probs[pos - 1]
        ent = -(p * p.clamp(min=1e-10).log()).sum().item()
        entropies.append(ent)

        # Rank of target token
        rank = (log_probs[pos - 1] > lp).sum().item() + 1
        ranks.append(rank)

        top1_correct.append(
            1.0 if log_probs[pos - 1].argmax().item() == target_id else 0.0
        )

    losses = np.array(losses)
    entropies = np.array(entropies)
    ranks = np.array(ranks)
    comp_ids = completion_ids[: len(losses)]

    # Text-level features
    text = sp_tok.decode(comp_ids)
    words = text.split()
    unique_tokens = len(set(comp_ids)) / max(len(comp_ids), 1)

    # Bigram repetition: fraction of token bigrams that are repeated
    bigrams = [(comp_ids[i], comp_ids[i + 1]) for i in range(len(comp_ids) - 1)]
    bigram_rep = 1.0 - len(set(bigrams)) / max(len(bigrams), 1)

    # Token appeared earlier in completion
    seen = set()
    n_repeated = 0
    for tid in comp_ids:
        if tid in seen:
            n_repeated += 1
        seen.add(tid)
    repeat_frac = n_repeated / max(len(comp_ids), 1)

    feats = {
        # Loss statistics
        "loss_mean": losses.mean(),
        "loss_std": losses.std(),
        "loss_max": losses.max(),
        "loss_min": losses.min(),
        "loss_median": np.median(losses),
        # Entropy statistics
        "entropy_mean": entropies.mean(),
        "entropy_std": entropies.std(),
        # Rank statistics
        "rank_mean": np.mean(ranks),
        "rank_median": np.median(ranks),
        "top1_accuracy": np.mean(top1_correct),
        # Consistency: loss vs entropy gap (overconfident errors)
        "loss_entropy_gap": (losses - entropies).mean(),
        # Text features
        "unique_token_ratio": unique_tokens,
        "bigram_repetition": bigram_rep,
        "token_repeat_frac": repeat_frac,
        "n_tokens": len(comp_ids),
        # Loss dynamics
        "loss_trend": np.polyfit(range(len(losses)), losses, 1)[0]
        if len(losses) > 1
        else 0,
        "loss_autocorr": np.corrcoef(losses[:-1], losses[1:])[0, 1]
        if len(losses) > 2
        else 0,
    }
    return feats


# Extract features for all real and generated completions
all_feats = []
all_labels = []  # 0=real, 1=generated

for d in tqdm(dataset, desc="Extracting features"):
    # Real
    f_real = extract_features(d["prompt_ids"], d["real_ids"])
    all_feats.append(f_real)
    all_labels.append(0)
    # Generated
    f_gen = extract_features(d["prompt_ids"], d["gen_ids"])
    all_feats.append(f_gen)
    all_labels.append(1)

feat_df = pd.DataFrame(all_feats)
feat_df["label"] = all_labels
feat_df["label_str"] = feat_df["label"].map({0: "real", 1: "generated"})

print(f"\nFeature matrix: {feat_df.shape}")
print(f"\nFeature means by class:")
print(
    feat_df.groupby("label_str")[feat_df.columns[:17]]
    .mean()
    .T.to_string(float_format="%.4f")
)

In [ ]:
# --- Train classifiers ---
feature_cols = [c for c in feat_df.columns if c not in ("label", "label_str")]
X = feat_df[feature_cols].values
y = feat_df["label"].values

# Handle any NaN from short sequences
X = np.nan_to_num(X, nan=0.0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, max_depth=6, random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100, max_depth=3, random_state=42
    ),
}

print("5-fold CV accuracy:")
best_name, best_score, best_clf = None, 0, None
for name, clf in classifiers.items():
    scores = cross_val_score(clf, X_scaled, y, cv=cv, scoring="accuracy")
    auc_scores = cross_val_score(clf, X_scaled, y, cv=cv, scoring="roc_auc")
    print(
        f"  {name:25s}: acc={scores.mean():.3f}±{scores.std():.3f}  AUC={auc_scores.mean():.3f}±{auc_scores.std():.3f}"
    )
    if auc_scores.mean() > best_score:
        best_name, best_score, best_clf = name, auc_scores.mean(), clf

print(f"\nBest: {best_name} (AUC={best_score:.3f})")

# Fit best on all data for analysis
best_clf.fit(X_scaled, y)
y_proba = best_clf.predict_proba(X_scaled)[:, 1]
y_pred = best_clf.predict(X_scaled)
print(f"\nFull-data classification report ({best_name}):")
print(classification_report(y, y_pred, target_names=["real", "generated"]))

In [ ]:
# --- Feature importance and analysis plots ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Feature importance (from best classifier)
ax = axes[0, 0]
if hasattr(best_clf, "feature_importances_"):
    importances = best_clf.feature_importances_
elif hasattr(best_clf, "coef_"):
    importances = np.abs(best_clf.coef_[0])
else:
    importances = np.zeros(len(feature_cols))
order = np.argsort(importances)[::-1]
top_n = min(12, len(feature_cols))
ax.barh(range(top_n), importances[order[:top_n]], color="steelblue")
ax.set_yticks(range(top_n))
ax.set_yticklabels([feature_cols[i] for i in order[:top_n]], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("Importance")
ax.set_title(f"Feature Importance ({best_name})")

# 2. ROC curve
ax = axes[0, 1]
fpr, tpr, _ = roc_curve(y, y_proba)
auc_val = roc_auc_score(y, y_proba)
ax.plot(fpr, tpr, color="steelblue", lw=2, label=f"AUC = {auc_val:.3f}")
ax.plot([0, 1], [0, 1], "--", color="gray")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve (full data)")
ax.legend()

# 3. Distribution of top 2 features by class
ax = axes[1, 0]
top2 = [feature_cols[i] for i in order[:2]]
real_mask = feat_df["label"] == 0
ax.scatter(
    feat_df.loc[real_mask, top2[0]],
    feat_df.loc[real_mask, top2[1]],
    alpha=0.3,
    s=10,
    label="real",
    color="steelblue",
)
ax.scatter(
    feat_df.loc[~real_mask, top2[0]],
    feat_df.loc[~real_mask, top2[1]],
    alpha=0.3,
    s=10,
    label="generated",
    color="coral",
)
ax.set_xlabel(top2[0])
ax.set_ylabel(top2[1])
ax.set_title("Top 2 features")
ax.legend()

# 4. Distribution of key features
ax = axes[1, 1]
key_feats = ["loss_mean", "entropy_mean", "bigram_repetition", "unique_token_ratio"]
key_feats = [f for f in key_feats if f in feature_cols]
x_pos = np.arange(len(key_feats))
real_means = [feat_df.loc[real_mask, f].mean() for f in key_feats]
gen_means = [feat_df.loc[~real_mask, f].mean() for f in key_feats]
bar_w = 0.35
ax.bar(x_pos - bar_w / 2, real_means, bar_w, label="real", color="steelblue")
ax.bar(x_pos + bar_w / 2, gen_means, bar_w, label="generated", color="coral")
ax.set_xticks(x_pos)
ax.set_xticklabels(key_feats, rotation=30, ha="right", fontsize=9)
ax.set_title("Key Features: Real vs Generated")
ax.legend()

plt.tight_layout()
plt.savefig(PLOT_DIR / "real_vs_generated_classifier.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: real_vs_generated_classifier.png")

In [ ]:
# --- Hardest examples: where does the classifier fail? ---
# Show cases where the classifier is most wrong — these are the most
# human-like generated text or most surprising real text.

feat_df["proba_gen"] = y_proba

# Generated text classified as real (most convincing fakes)
gen_mask = feat_df["label"] == 1
convincing_fakes = feat_df[gen_mask].nsmallest(5, "proba_gen")

print("=" * 70)
print("Most convincing GENERATED text (classifier thought was real):")
print("=" * 70)
for idx in convincing_fakes.index:
    pair_idx = idx // 2
    d = dataset[pair_idx]
    print(f"\n  P(gen)={feat_df.loc[idx, 'proba_gen']:.3f}")
    print(f"  Prompt: {d['prompt_text'][:80]}...")
    print(f"  Generated: {d['gen_text'][:150]}")

# Real text classified as generated (most surprising real text)
real_mask = feat_df["label"] == 0
surprising_real = feat_df[real_mask].nlargest(5, "proba_gen")

print(f"\n{'=' * 70}")
print("Most surprising REAL text (classifier thought was generated):")
print("=" * 70)
for idx in surprising_real.index:
    pair_idx = idx // 2
    d = dataset[pair_idx]
    print(f"\n  P(gen)={feat_df.loc[idx, 'proba_gen']:.3f}")
    print(f"  Prompt: {d['prompt_text'][:80]}...")
    print(f"  Real: {d['real_text'][:150]}")

In [ ]:
# --- Tiny transformer classifier (for fun) ---
# Train a 1-layer transformer directly on token IDs to classify real vs generated.
# This tests whether the signal is learnable end-to-end, not just from hand-picked features.


class TinyClassifier(torch.nn.Module):
    def __init__(self, vocab_size, dim=64, max_len=64, n_heads=4):
        super().__init__()
        self.tok_emb = torch.nn.Embedding(vocab_size, dim)
        self.pos_emb = torch.nn.Embedding(max_len, dim)
        encoder_layer = torch.nn.TransformerEncoderLayer(
            d_model=dim,
            nhead=n_heads,
            dim_feedforward=dim * 4,
            dropout=0.1,
            batch_first=True,
        )
        self.encoder = torch.nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.head = torch.nn.Linear(dim, 1)

    def forward(self, x):
        B, L = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(0)
        h = self.tok_emb(x) + self.pos_emb(pos)
        h = self.encoder(h)
        h = h.mean(dim=1)  # mean pool
        return self.head(h).squeeze(-1)


# Prepare data: pad/truncate completions to fixed length
MAX_CLS_LEN = 40
X_ids = []
y_cls = []
for d in dataset:
    for ids, label in [(d["real_ids"], 0), (d["gen_ids"], 1)]:
        padded = (ids + [0] * MAX_CLS_LEN)[:MAX_CLS_LEN]
        X_ids.append(padded)
        y_cls.append(label)

X_ids = torch.tensor(X_ids)
y_cls = torch.tensor(y_cls, dtype=torch.float32)
n_total = len(y_cls)

# Train/val split
perm = torch.randperm(n_total)
n_train = int(0.8 * n_total)
train_idx, val_idx = perm[:n_train], perm[n_train:]

cls_model = TinyClassifier(V, dim=64, max_len=MAX_CLS_LEN, n_heads=4)
optimizer = torch.optim.Adam(cls_model.parameters(), lr=1e-3)
loss_fn = torch.nn.BCEWithLogitsLoss()

# Training loop
train_losses = []
val_accs = []
BATCH = 64
EPOCHS = 30

for epoch in range(EPOCHS):
    cls_model.train()
    epoch_loss = 0
    shuffled = train_idx[torch.randperm(len(train_idx))]
    for i in range(0, len(shuffled), BATCH):
        batch_i = shuffled[i : i + BATCH]
        logits = cls_model(X_ids[batch_i])
        loss = loss_fn(logits, y_cls[batch_i])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(batch_i)
    train_losses.append(epoch_loss / len(train_idx))

    # Validation
    cls_model.eval()
    with torch.no_grad():
        val_logits = cls_model(X_ids[val_idx])
        val_preds = (val_logits > 0).float()
        val_acc = (val_preds == y_cls[val_idx]).float().mean().item()
        val_accs.append(val_acc)

    if (epoch + 1) % 5 == 0:
        print(
            f"Epoch {epoch + 1:3d}: train_loss={train_losses[-1]:.4f}  val_acc={val_acc:.3f}"
        )

# Plot training
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Training Loss")
ax2.plot(val_accs)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Validation Accuracy")
ax2.axhline(0.5, color="gray", ls="--", label="chance")
ax2.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "real_vs_gen_transformer_training.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\nFinal val accuracy: {val_accs[-1]:.3f}")
print(f"Best val accuracy:  {max(val_accs):.3f} (epoch {np.argmax(val_accs) + 1})")
print("Saved: real_vs_gen_transformer_training.png")